In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import top_k_accuracy_score

import h5py

In [ ]:
def load_bin_events(path):
  raw = np.fromfile(path, dtype=np.uint8)
  raw = raw[:len(raw) // 5 * 5]
  x = raw[0::5].astype(np.int32)
  y = raw[1::5].astype(np.int32)

  p = (raw[2::5] >> 7) & 1
  p = np.where(p == 1, 1, -1)

  t_high = (raw[2::5] & 0x7F).astype(np.uint32)
  t_mid  = raw[3::5].astype(np.uint32)
  t_low  = raw[4::5].astype(np.uint32)
  t = (t_high << 16) | (t_mid << 8) | t_low

  return x, y, t.astype(np.float32), p

class EventDatasetBIN(Dataset):
  def __init__(self, root_dir, classes, T=10, H=180, W=240, clip_value=5.0, augment=False):
    self.root_dir = Path(root_dir)
    self.classes = classes
    self.class_to_idx = {c: i for i, c in enumerate(classes)}
    self.T, self.H, self.W = T, H, W
    self.clip_value = clip_value
    self.augment = augment

    self.samples = []
    for cls in classes:
        for f in (self.root_dir / cls).glob("*.bin"):
            self.samples.append((f, self.class_to_idx[cls]))

  def __len__(self):
      return len(self.samples)

  def __getitem__(self, idx):
      path, label = self.samples[idx]
      x, y, t, p = load_bin_events(path)

      if self.augment and np.random.rand() > 0.5:
          x = (self.W - 1) - x

      t = (t - t.min()) / (t.max() - t.min() + 1e-9)
      voxels = np.zeros((2, self.H, self.W, self.T), dtype=np.float32)
      time_bins = (t * (self.T - 1)).astype(np.int32)

      for xi, yi, pi, bi in zip(x, y, p, time_bins):
          if 0 <= xi < self.W and 0 <= yi < self.H:
              ch = 0 if pi > 0 else 1
              voxels[ch, yi, xi, bi] += 1.0

      voxels = np.clip(voxels, 0.0, self.clip_value)
      voxels /= self.clip_value

      return torch.from_numpy(voxels), torch.tensor(label)

In [ ]:
class SimpleSNN(nn.Module):
    def __init__(self, num_classes=5, T=10): # num_classes=5 domyślnie, ale zostanie nadpisane przez len(CLASSES)
        super().__init__()
        self.T = T
        beta = 0.9
        spike_grad = surrogate.fast_sigmoid()

        self.conv1 = nn.Conv2d(2, 16, 3, stride=2, padding=1)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        with torch.no_grad():
            dummy = torch.zeros(1, 2, 180, 240)
            x = self.lif1(self.conv1(dummy))[0]
            x = self.lif2(self.conv2(x))[0]
            self.flat_dim = x.numel()

        self.fc = nn.Linear(self.flat_dim, num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem_out = self.lif_out.init_leaky()

        spk_rec = []
        for t in range(self.T):
            xt = x[..., t]

            spk1, mem1 = self.lif1(self.conv1(xt), mem1)
            spk2, mem2 = self.lif2(self.conv2(spk1), mem2)
            flat = spk2.flatten(1)
            spk_out, mem_out = self.lif_out(self.fc(flat), mem_out)
            spk_rec.append(spk_out)

        return torch.stack(spk_rec)

In [ ]:
def evaluate(model, loader, device, criterion, classes):
  model.eval()
  total_loss = 0.0
  correct = 0
  total = 0

  all_labels = []
  all_probs = []
  spike_rates = []

  with torch.no_grad():
      for x, y in loader:
          x, y = x.to(device), y.to(device)

          spk = model(x)
          sum_spk = spk.sum(dim=0)

          loss = criterion(sum_spk, y)
          total_loss += loss.item()

          probs = F.softmax(sum_spk, dim=1)
          preds = probs.argmax(dim=1)

          correct += (preds == y).sum().item()
          total += y.size(0)

          rate = spk.mean(dim=(0, 2))

          all_labels.extend(y.cpu().numpy())
          all_probs.extend(probs.cpu().numpy())
          spike_rates.append(rate.cpu())

  val_loss = total_loss / len(loader)
  val_acc = correct / total

  all_probs = np.array(all_probs)
  all_labels = np.array(all_labels)

  top1 = top_k_accuracy_score(all_labels, all_probs, k=1)
  top5 = top_k_accuracy_score(all_labels, all_probs, k=5)

  spike_rates = torch.cat(spike_rates)

  return (
      val_loss,
      val_acc,
      top1,
      top5,
      all_labels,
      all_probs,
      spike_rates
      )

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

TRAIN_ROOT = "/content/N-Caltech101-full-split/N-Caltech101-full-split/train"
TEST_ROOT  = "/content/N-Caltech101-full-split/N-Caltech101-full-split/test"
SAVE_DIR   = Path("/content/results")
SAVE_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = SAVE_DIR / "checkpoint.pth"

CLASSES = sorted([p.name for p in Path(TRAIN_ROOT).iterdir()])
NUM_CLASSES = len(CLASSES)
print(NUM_CLASSES)
train_ds = EventDatasetBIN(TRAIN_ROOT, CLASSES, augment=True)
test_ds  = EventDatasetBIN(TEST_ROOT, CLASSES)
labels = [s[1] for s in train_ds.samples]
counts = Counter(labels)
weights = [1 / counts[l] for l in labels]
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers = 2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=64, num_workers = 2, pin_memory=True)

model = SimpleSNN(num_classes=NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scaler = torch.amp.GradScaler(
  enabled=(device.type == "cuda")
  )

criterion = nn.CrossEntropyLoss()

In [ ]:
EPOCHS = 10

history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "top1": [],
    "top5": [],
    "spike_rate": []
}
best_val_acc = 0.0
start_epoch = 0

if CHECKPOINT_PATH.exists():
    print("🔁 Wczytuję checkpoint...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    history = checkpoint["history"]
    best_val_acc = checkpoint["best_val_acc"]
    start_epoch = checkpoint["epoch"] + 1 # Wznawiamy od następnej epoki

    last_train_acc = history["train_acc"][-1] if history["train_acc"] else 0.0
    last_val_acc   = history["val_acc"][-1] if history["val_acc"] else 0.0
    last_top1      = history["top1"][-1] if history["top1"] else 0.0
    last_top5      = history["top5"][-1] if history["top5"] else 0.0

    print(f"🔁 Wczytano checkpoint z epoki {start_epoch if start_epoch > 0 else 0}")
    print(f"Train Acc: {last_train_acc*100:.2f}%")
    print(f"Val Acc:   {last_val_acc*100:.2f}%")
    print(f"Top-1:     {last_top1*100:.2f}%")
    print(f"Top-5:     {last_top5*100:.2f}%")

    print(f"➡️ Wznawiam od epoki {start_epoch}")
else:
    print("🆕 Brak checkpointu – start od zera")

print(f"Dane: Trening ({len(train_ds)}), Test ({len(test_ds)})")
print(f"Rozpoczynam trening. Wyniki w: {SAVE_DIR.absolute()}")

for epoch in range(start_epoch, EPOCHS):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)


        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            spk = model(x)
            sum_spk = spk.sum(dim=0)
            loss = criterion(sum_spk, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds = sum_spk.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    (
        val_loss,
        val_acc,
        top1,
        top5,
        labels,
        probs,
        spike_rates
    ) = evaluate(
        model,
        test_loader,
        device,
        criterion,
        CLASSES
    )

    history["epoch"].append(epoch+1)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["top1"].append(top1)
    history["top5"].append(top5)
    avg_spike_rate = spike_rates.mean().item()
    history["spike_rate"].append(avg_spike_rate)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")

    print(
        f"Epoch {epoch+1}: "
        f"Train Acc={train_acc*100:.3f}%, "
        f"Val Acc={val_acc*100:.3f}%, "
        f"Top1={top1*100:.3f}%, "
        f"Top5={top5*100:.3f}%"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "history": history
        },
        CHECKPOINT_PATH
    )
